# Etapa 3 – Estatísticas descritivas e contagem de regimes

Lê `dados/processados/base_diaria.csv` (gerada por `02_processamento.ipynb`) e produz:

- `resultados/tabela_descritiva.csv`: média, desvio-padrão, mínimo, máximo e número de observações dos retornos diários e do VIX, na amostra toda e em cada regime;
- `resultados/contagem_regimes.csv`: dias e número de episódios em cada regime de crise;
- `resultados/descritivas.md`: as duas tabelas formatadas, prontas para conferência e para a seção de resultados.

Retornos em % ao dia (log-retorno × 100). Este notebook só descreve os dados; os testes das proposições ficam para a etapa 4.

In [1]:
from pathlib import Path

import pandas as pd

RAIZ = Path.cwd().parent if Path.cwd().name == "codigo" else Path.cwd()
PASTA_RESULTADOS = RAIZ / "resultados"

base = pd.read_csv(RAIZ / "dados" / "processados" / "base_diaria.csv", parse_dates=["data"], index_col="data")
print(f"Base: {base.index.min().date()} a {base.index.max().date()}, {len(base)} dias")

SERIES = {
    "r_sp500": "S&P 500",
    "r_treasury": "Treasury 10 anos",
    "r_ibov": "Ibovespa",
    "r_ntnb": "IMA-B (NTN-B)",
    "r_usdbrl": "USD/BRL (PTAX)",
    "vix": "VIX (nível)",
    "r_tips": "TIPS 10 anos",
    "r_ief": "ETF IEF",
    "r_ntnb5mais": "IMA-B 5+",
    "r_idka10": "IDkA IPCA 10A",
    "r_irfm": "IRF-M",
    "r_usdbrl_fech": "USD/BRL (fechamento)",
}

REGIMES = {
    "Amostra toda": None,
    "Normal": "normal",
    "Só crise global": "so_global",
    "Só crise local (BR)": "so_local",
    "Ambas": "ambas",
    "Crise EUA (P1)": "crise_p1_eua",
}

Base: 2003-09-16 a 2026-09-30, 5526 dias


## 1. Tabela descritiva por série e por regime

Os regimes "Normal", "Só crise global", "Só crise local" e "Ambas" são mutuamente exclusivos e somam a amostra toda. "Crise EUA (P1)" (VIX > 30 ou drawdown do S&P 500 ≤ −20%) se sobrepõe aos demais.

In [2]:
linhas = []
for nome_regime, coluna in REGIMES.items():
    dados = base if coluna is None else base[base[coluna] == 1]
    for serie, rotulo in SERIES.items():
        x = dados[serie].dropna()
        linhas.append({"regime": nome_regime, "série": rotulo, "n": len(x),
                       "média": x.mean(), "desvio": x.std(), "mínimo": x.min(), "máximo": x.max()})

descritiva = pd.DataFrame(linhas)
descritiva[["média", "desvio", "mínimo", "máximo"]] = descritiva[["média", "desvio", "mínimo", "máximo"]].round(3)

principais = list(SERIES.values())[:6]
descritiva[descritiva["série"].isin(principais)].pivot_table(
    index="série", columns="regime", values="desvio", sort=False
)[list(REGIMES)]

regime,Amostra toda,Normal,Só crise global,Só crise local (BR),Ambas,Crise EUA (P1)
série,,,,,,
S&P 500,1.191,0.860,2.160,1.167,3.204,2.731
Treasury 10 anos,0.490,0.442,0.677,0.524,0.886,0.808
Ibovespa,1.657,1.374,1.799,1.744,3.853,3.164
IMA-B (NTN-B),0.411,0.360,0.394,0.553,0.760,0.628
USD/BRL (PTAX),0.930,0.784,1.236,1.100,1.954,1.676
VIX (nível),8.303,4.522,3.975,4.898,11.818,11.617


## 2. Contagem de dias e episódios por regime

Um **episódio** é uma sequência de dias consecutivos (no calendário comum) dentro do regime. O número de episódios mostra se os dias de um regime vêm de muitos eventos ou de poucos eventos longos, o que importa para a P4: correlações de dias do mesmo episódio não são independentes.

In [3]:
CONTAGEM = {
    "crise_global": "VIX > 30",
    "crise_global_p90": "VIX > percentil 90",
    "crise_local_br": "Drawdown do Ibovespa ≤ −20%",
    "crise_eua": "Drawdown do S&P 500 ≤ −20%",
    "crise_p1_eua": "Crise EUA para a P1 (VIX > 30 ou drawdown do S&P 500)",
    "normal": "Normal",
    "so_global": "Só crise global",
    "so_local": "Só crise local",
    "ambas": "Crise local e global",
    "so_local_p90": "Só crise local (VIX p90) – robustez",
    "so_local_sem_cauda": "Só crise local sem cauda de crise global – robustez (a)",
    "crise_local_fixa": "Lista fixa de eventos locais – robustez (b)",
    "choque_local_br": "Choque de 21 pregões no Ibovespa",
    "so_local_choque": "Choque local sem crise global – robustez (c)",
}

def episodios(indicador):
    return int(((indicador == 1) & (indicador.shift(1, fill_value=0) == 0)).sum())

contagem = pd.DataFrame([
    {"regime": coluna, "descrição": desc, "dias": int(base[coluna].sum()),
     "% da amostra": round(100 * base[coluna].mean(), 1), "episódios": episodios(base[coluna]),
     "maior episódio (dias)": int(base[coluna].groupby((base[coluna] != base[coluna].shift()).cumsum()).sum().max())}
    for coluna, desc in CONTAGEM.items()
])
contagem

,regime,descrição,dias,% da amostra,episódios,maior episódio (dias)
0,crise_global,VIX > 30,422,7.6,51,161
1,crise_global_p90,VIX > percentil 90,553,10.0,71,186
2,crise_local_br,Drawdown do Ibovespa ≤ −20%,729,13.2,59,222
3,crise_eua,Drawdown do S&P 500 ≤ −20%,314,5.7,20,225
4,crise_p1_eua,Crise EUA para a P1 (VIX > 30 ou drawdown do S...,524,9.5,57,234
5,normal,Normal,4686,84.8,81,482
6,so_global,Só crise global,111,2.0,39,10
7,so_local,Só crise local,418,7.6,66,60
8,ambas,Crise local e global,311,5.6,29,161
9,so_local_p90,Só crise local (VIX p90) – robustez,378,6.8,58,46


## 3. Ponto crítico da P4

O plano pede para rever a P4 se houver poucos dias de "só crise local". A célula abaixo registra a contagem nas três definições e o número de episódios.

In [4]:
p4 = contagem.set_index("regime").loc[["so_local", "so_local_sem_cauda", "so_local_choque", "crise_local_fixa", "so_global"],
                                       ["dias", "episódios", "maior episódio (dias)"]]
p4

,dias,episódios,maior episódio (dias)
regime,,,
so_local,418,66,60
so_local_sem_cauda,296,36,53
so_local_choque,182,48,19
crise_local_fixa,84,4,21
so_global,111,39,10


## 4. Saídas

In [5]:
PASTA_RESULTADOS.mkdir(exist_ok=True)
descritiva.to_csv(PASTA_RESULTADOS / "tabela_descritiva.csv", index=False)
contagem.to_csv(PASTA_RESULTADOS / "contagem_regimes.csv", index=False)

texto = [
    "# Estatísticas descritivas e contagem de regimes", "",
    f"Gerado por `codigo/03_descritivas.ipynb` a partir de `dados/processados/base_diaria.csv` "
    f"({base.index.min().date()} a {base.index.max().date()}, {len(base)} dias úteis comuns). "
    "Retornos em % ao dia (log-retorno × 100); VIX em pontos.", "",
    "## Contagem de dias e episódios por regime", "", contagem.to_markdown(index=False), "",
]
for nome_regime in REGIMES:
    tabela = descritiva[descritiva["regime"] == nome_regime].drop(columns="regime")
    texto += [f"## Descritivas – {nome_regime}", "", tabela.to_markdown(index=False), ""]
(PASTA_RESULTADOS / "descritivas.md").write_text("\n".join(texto), encoding="utf-8")
print("resultados/: tabela_descritiva.csv, contagem_regimes.csv, descritivas.md")

resultados/: tabela_descritiva.csv, contagem_regimes.csv, descritivas.md
